# AGILAB API first run in Kaggle

This notebook shows the smallest published-package AGILAB API path from Kaggle.
It installs AGILAB from PyPI and runs the built-in `mycode_project` locally
through `AgiEnv` and `AGI.run(...)`.


Kaggle Internet must be enabled for the first install.
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_kaggle_first_run.ipynb)


In [ ]:
# Published-package Kaggle path: install AGILAB from PyPI.
import json
import os
import subprocess
import sys
import socket
from pathlib import Path

STATE_FILE = Path("/kaggle/working/.agilab_kaggle_pypi_state.json")
STATE_VERSION = 2
SENSITIVE_PREFIXES = (
    "agilab.",
    "agi_env", "agi_env.",
    "agi_node", "agi_node.",
    "agi_cluster", "agi_cluster.",
    "numpy", "numpy.",
    "scipy", "scipy.",
    "pandas", "pandas.",
    "polars", "polars.",
    "numba", "numba.",
)

def require_kaggle_internet(*hosts):
    missing = []
    for host in hosts:
        try:
            socket.getaddrinfo(host, 443)
        except OSError:
            missing.append(host)
    if missing:
        hosts_text = ", ".join(missing)
        raise SystemExit(
            f"Kaggle Internet appears to be disabled for this notebook session; cannot reach {hosts_text}. Enable Internet in the Kaggle notebook settings, then rerun Cell 1."
        )

require_kaggle_internet("pypi.org")

state = {}
if STATE_FILE.is_file():
    try:
        state = json.loads(STATE_FILE.read_text())
    except Exception:
        state = {}

needs_install = (
    state.get("version") != STATE_VERSION
    or state.get("mode") != "pypi"
)

if needs_install:
    restart_required = any(name == "agilab" or name.startswith(SENSITIVE_PREFIXES) for name in sys.modules)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "agilab", "agi-core", "agi-cluster", "agi-node", "agi-env"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir",
        "uv", "agi-core", "agilab",
    ], check=True)
    STATE_FILE.write_text(json.dumps({
        "version": STATE_VERSION,
        "mode": "pypi",
        "install_pid": os.getpid(),
        "restart_required": restart_required,
    }))
    if restart_required:
        raise SystemExit("AGILAB PyPI packages were updated after scientific or AGILAB modules were already loaded. Restart the Kaggle session, then rerun the notebook from the top.")
    print("AGILAB PyPI packages installed in a fresh Kaggle session; continuing without restart.")
else:
    print("AGILAB PyPI packages already prepared for this Kaggle session.")


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

STATE_FILE = Path("/kaggle/working/.agilab_kaggle_pypi_state.json")
if STATE_FILE.is_file():
    try:
        _kaggle_state = json.loads(STATE_FILE.read_text())
    except Exception:
        _kaggle_state = {}
    if _kaggle_state.get("install_pid") == os.getpid() and _kaggle_state.get("restart_required"):
        raise RuntimeError("AGILAB PyPI packages were installed in this same Kaggle session after scientific or AGILAB modules were already loaded. Restart the session, then rerun the notebook from the top.")

for bad_prefix in ("/kaggle/working/agilab/src", "/kaggle/working/agilab"):
    sys.path = [p for p in sys.path if p != bad_prefix and not p.startswith(bad_prefix + "/")]
for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

import agilab
from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

APPS_PATH = Path(agilab.__file__).resolve().parent / "apps"
BUILTIN_ROOT = APPS_PATH / "builtin"
APP = "mycode_project"
LOG_ROOT = Path.home() / "log" / "execute" / "mycode"

def worker_env_ready(app_env):
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        return False
    cmd = ["uv", "--quiet", "run", "--no-sync", "--project", str(worker_venv.parent)]
    pyvers_worker = getattr(app_env, "pyvers_worker", None)
    if pyvers_worker:
        cmd.extend(["--python", str(pyvers_worker)])
    cmd.extend(["python", "-c", "import agi_env, agi_node"])
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    return result.returncode == 0

async def install_if_needed(app_env, *, scheduler="127.0.0.1", workers=None, modes_enabled=0):
    if workers is None:
        workers = {"127.0.0.1": 1}
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if worker_env_ready(app_env):
        return False
    action = "Installing"
    if worker_venv.parent.exists():
        shutil.rmtree(worker_venv.parent, ignore_errors=True)
        action = "Reinstalling"
    print(f"{action} worker for {app_env.app}...")
    await AGI.install(app_env, scheduler=scheduler, workers=workers, modes_enabled=modes_enabled)
    return True

print("Installed apps path:", APPS_PATH)
print("Builtin root:", BUILTIN_ROOT)
print("App:", APP)
print("Log root:", LOG_ROOT)


In [ ]:
app_env = AgiEnv(apps_path=APPS_PATH, app=APP, verbose=0)
await install_if_needed(app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
result = await AGI.run(
    app_env,
    scheduler="127.0.0.1",
    workers={"127.0.0.1": 1},
    mode=0,
)
result
